In [14]:
# Falls nicht installiert:
# pip install xarray netCDF4 pandas numpy scipy

import xarray as xr
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

In [15]:
spots = pd.read_csv("spots.csv")
spots.head()

,name,country,region,lat,lon
0,Great Barrier Reef - Osprey Reef,Australia,Indo-Pacific,-13.8833,146.5667
1,Great Barrier Reef - Cod Hole,Australia,Indo-Pacific,-16.1167,145.9833
2,Coral Sea - Holmes Reef,Australia,Indo-Pacific,-16.4833,147.8667
3,Ribbon Reefs - No. 10,Australia,Indo-Pacific,-15.0833,145.7500
4,Raja Ampat - Misool,Indonesia,Indo-Pacific,-2.0833,130.1667


In [16]:
# read NOAA data
ds_stress = xr.open_dataset("noaa_crw_thermal_history_stress_freq_v3.7.0.nc")
ds_trend  = xr.open_dataset("noaa_crw_thermal_history_sst_trend_v3.7.0.nc")

# have a first look at the data
print("=== STRESS FREQUENCY ===")
print(ds_stress)
print("=== SST TREND ===")
print(ds_trend)

=== STRESS FREQUENCY ===
<xarray.Dataset> Size: 420MB
Dimensions:    (lat: 1390, lon: 7200, years: 41)
Coordinates:
  * lat        (lat) float64 11kB -35.27 -35.22 -35.17 ... 34.08 34.12 34.18
  * lon        (lon) float64 58kB -180.0 -179.9 -179.9 ... 179.9 179.9 180.0
  * years      (years) int32 164B 1985 1986 1987 1988 ... 2022 2023 2024 2025
Data variables: (12/13)
    reef_mask  (lat, lon) int8 10MB ...
    mask       (lat, lon) int8 10MB ...
    n_gt0      (lat, lon) int32 40MB ...
    n_ge2      (lat, lon) int32 40MB ...
    n_ge4      (lat, lon) int32 40MB ...
    n_ge6      (lat, lon) int32 40MB ...
    ...         ...
    rp_gt0     (lat, lon) float32 40MB ...
    rp_ge2     (lat, lon) float32 40MB ...
    rp_ge4     (lat, lon) float32 40MB ...
    rp_ge6     (lat, lon) float32 40MB ...
    rp_ge8     (lat, lon) float32 40MB ...
    crs        int16 2B ...
Attributes: (12/61)
    conventions:                    CF-1.6, ACDD 1-3, Unidata Observation Dat...
    nodc_template_ve

In [17]:
# Identify variable names
print("Stress Frequency Variablen:", list(ds_stress.data_vars))
print("SST Trend Variablen:", list(ds_trend.data_vars))

Stress Frequency Variablen: ['reef_mask', 'mask', 'n_gt0', 'n_ge2', 'n_ge4', 'n_ge6', 'n_ge8', 'rp_gt0', 'rp_ge2', 'rp_ge4', 'rp_ge6', 'rp_ge8', 'crs']
SST Trend Variablen: ['reef_mask', 'mask', 'trend_warmseason', 'trend_annual', 'trend_maxmonth', 'crs']


In [18]:
# Nearest Neighbor Join
VAR_STRESS = "n_ge4"           # Anzahl signifikante Bleaching-Events (DHW >= 4)
VAR_SEVERE = "n_ge8"           # Anzahl severe Events (DHW >= 8)
VAR_TREND  = "trend_warmseason" # SST Trend °C/Dekade

# Koordinaten aus den Datasets extrahieren
lats_stress = ds_stress["lat"].values
lons_stress  = ds_stress["lon"].values

# 2D Raster-Gitter aufbauen
lon_grid, lat_grid = np.meshgrid(lons_stress, lats_stress)
raster_coords = np.column_stack([lat_grid.ravel(), lon_grid.ravel()])

# KD-Tree für schnellen Nearest-Neighbor
tree = cKDTree(raster_coords)

# Für jeden Spot den nächsten Rasterpunkt finden
spot_coords = spots[["lat", "lon"]].values
_, indices = tree.query(spot_coords)

# Flat indices → 2D row/col
nrows = len(lats_stress)
ncols = len(lons_stress)
row_idx = indices // ncols
col_idx = indices % ncols

In [22]:
# Alle gültigen Riff-Pixel aus dem Stress-Dataset extrahieren
reef_mask = ds_stress["reef_mask"].values  # shape: (lat, lon)

# Nur Pixel wo reef_mask == 1
reef_rows, reef_cols = np.where(reef_mask == 1)
reef_lats = lats_stress[reef_rows]
reef_lons = lons_stress[reef_cols]
reef_coords = np.column_stack([reef_lats, reef_lons])

# Neuer KD-Tree nur auf Riff-Pixeln
reef_tree = cKDTree(reef_coords)

# Für jeden Spot den nächsten REEF-Pixel finden
_, reef_indices = reef_tree.query(spot_coords)

# Zurück auf row/col im Original-Raster mappen
row_idx = reef_rows[reef_indices]
col_idx = reef_cols[reef_indices]

print("✓ Nearest reef pixel gefunden für alle Spots")

✓ Nearest reef pixel gefunden für alle Spots


In [23]:
# extract values and build panel dataset

results = []

for i, spot in spots.iterrows():
    r, c = row_idx[i], col_idx[i]
    
    try:
        stress_val = float(ds_stress[VAR_STRESS].values[r, c])
        severe_val = float(ds_stress[VAR_SEVERE].values[r, c])
        trend_val  = float(ds_trend[VAR_TREND].values[r, c])
    except Exception as e:
        stress_val, severe_val, trend_val = np.nan, np.nan, np.nan
        print(f"Fehler bei {spot['name']}: {e}")
    
    results.append({
        "name":                       spot["name"],
        "country":                    spot["country"],
        "region":                     spot["region"],
        "lat":                        spot["lat"],
        "lon":                        spot["lon"],
        "bleaching_freq_significant": stress_val,
        "bleaching_freq_severe":      severe_val,
        "sst_trend_per_decade":       trend_val,
    })

# ERST nach der Schleife:
panel = pd.DataFrame(results)

valid = panel.dropna()
print(f" {len(valid)}/{len(panel)} Spots mit gültigen NOAA-Werten")
print(f" {len(panel) - len(valid)} Spots außerhalb des Riff-Datensatzes (NaN)")
print(panel[panel.isna().any(axis=1)][["name", "country"]])

 49/50 Spots mit gültigen NOAA-Werten
 1 Spots außerhalb des Riff-Datensatzes (NaN)
                         name country
33  Red Sea - Blue Hole Dahab   Egypt


In [24]:
# Diagnose: warum NaN bei reef-nahen Spots?
for i, spot in spots.iterrows():
    if spot["name"] in ["Koh Tao - Chumphon Pinnacle", "Ari Atoll - Maaya Thila", "Red Sea - Brother Islands"]:
        r, c = row_idx[i], col_idx[i]
        print(f"\n{spot['name']}")
        print(f"  lat/lon: {spot['lat']}, {spot['lon']}")
        print(f"  reef_mask: {ds_stress['reef_mask'].values[r, c]}")
        print(f"  mask:      {ds_stress['mask'].values[r, c]}")
        print(f"  n_ge4:     {ds_stress['n_ge4'].values[r, c]}")


Koh Tao - Chumphon Pinnacle
  lat/lon: 10.6667, 99.8167
  reef_mask: 1
  mask:      1
  n_ge4:     3

Ari Atoll - Maaya Thila
  lat/lon: 3.9833, 72.5333
  reef_mask: 1
  mask:      1
  n_ge4:     4

Red Sea - Brother Islands
  lat/lon: 26.15, 34.8667
  reef_mask: 1
  mask:      1
  n_ge4:     11


In [25]:
panel.to_csv("panel_dataset.csv", index=False)
print("panel_dataset.csv gespeichert")
panel.describe()

panel_dataset.csv gespeichert


,lat,lon,bleaching_freq_significant,bleaching_freq_severe,sst_trend_per_decade
count,50.000000,50.000000,50.000000,50.000000,49.000000
mean,4.883336,52.720666,6.100000,2.220000,0.227797
std,15.980240,90.161552,4.362409,2.349902,0.081625
min,-31.550000,-156.483300,0.000000,0.000000,0.053161
25%,-5.733350,-10.454150,3.000000,1.000000,0.183649
50%,5.341650,85.583300,5.000000,2.000000,0.220908
75%,16.020850,124.050000,8.750000,3.000000,0.286835
max,38.233300,159.083300,20.000000,12.000000,0.479725
